# Modelo v5 para la web (Colab Pro, GPU A100 o L4)
Abrir en Colab: https://colab.research.google.com/github/JavierAnd0/ia-Kaggle-Coffee/blob/claude/modelo-v5/notebooks/entrenar_colab_v5.ipynb

**Cómo usarlo:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100 (o L4)* y luego *Entorno de ejecución → Ejecutar todas*. No hay que editar nada.
Pedirá permiso para montar Google Drive: ahí se guardan los modelos parciales (`MyDrive/cafe_v5/checkpoints`). Si Colab se desconecta, vuelva a *Ejecutar todas* y continúa desde la última semilla terminada.

**Qué hace** (todo con `scripts/entrenar_v5.py`):
1. Splits v5 (`data_splits/v5/`): se quitan las fotos duplicadas con etiquetas contradictorias; partición 70/15/15 estratificada por clase y agrupada por planta + duplicados.
2. EfficientNetV2-B0 a 224 px, semillas 0 a 4. Fase 1: solo la cabeza (lr 1e-3, 8 épocas). Fase 2: todo el backbone (AdamW, lr 1e-4 con coseno, hasta 30 épocas; BatchNorm en modo inferencia). Pesos de clase + label smoothing 0,1 (sin sobremuestreo). La mejor época se elige por **F1 macro de validación**.
3. **Antes de mirar test** decide qué exportar: el conjunto de las 5 semillas solo si su F1 macro de validación supera al mejor modelo individual por al menos 0,01.
4. Evalúa en **test una sola vez** (mejor individual y conjunto) y exporta a `web/model/` (ONNX + ficha + paridad + métricas para la web), comprobando Keras contra ONNX con fotos reales.
5. Guarda `MyDrive/cafe_v5/modelo_cafe_v5.zip` y lo descarga.

**Tiempo estimado:** A100 ≈ 30-45 min; L4 ≈ 50-80 min (incluye instalar y descargar los pesos de ImageNet). Con `FAST = True` (prueba de humo) ≈ 3-5 min.

In [ ]:
# Prueba de humo: True = pocas fotos, 2 semillas, 1+1 épocas, 128 px. Sus métricas NO sirven y no toca web/model.
FAST = False

BACKBONE = "efficientnetv2b0"     # o "convnext_tiny" (más lento y pesado; opcional)
RAMA = "claude/modelo-v5"
URL_REPO = "https://github.com/JavierAnd0/ia-Kaggle-Coffee"

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

# Si la carpeta actual fue borrada (p. ej. se eliminó el clon con el kernel dentro), se vuelve a una que exista.
try:
    os.getcwd()
except FileNotFoundError:
    os.chdir("/content" if os.path.exists("/content") else str(Path.home()))

EN_COLAB = os.path.exists("/content")


def clonar_o_actualizar(destino, rama):
    """Clona la rama (con reintentos) o, si ya estaba clonada en esta sesión, la deja en la última versión.
    Si la rama ya no existe (p. ej. tras fusionar el PR), usa main."""
    if os.path.isdir(os.path.join(destino, ".git")):
        for r in (rama, "main"):
            f = subprocess.run(["git", "-C", destino, "fetch", "-q", "origin", r], capture_output=True, text=True)
            if f.returncode == 0:
                subprocess.run(["git", "-C", destino, "checkout", "-q", "-B", r, f"origin/{r}"], check=True)
                subprocess.run(["git", "-C", destino, "reset", "-q", "--hard", f"origin/{r}"], check=True)
                return r
            print(f"No pude actualizar la rama {r}: {f.stderr.strip()}")
        raise SystemExit("No pude actualizar el repositorio; revise la conexión.")
    error = ""
    for r in (rama, "main"):
        for intento in range(1, 4):
            c = subprocess.run(["git", "clone", "-q", "-b", r, URL_REPO, destino], capture_output=True, text=True)
            if c.returncode == 0:
                return r
            error = c.stderr.strip()
            print(f"Intento {intento}/3 de clonar la rama {r} falló: {error}")
            if "not found" in error.lower():
                break                               # la rama no existe: probar main
            time.sleep(5 * intento)
    raise SystemExit(f"No pude clonar {URL_REPO}.\nDetalle de git: {error}\nCompruebe su conexión.")


if EN_COLAB:
    usada = clonar_o_actualizar("/content/ia-Kaggle-Coffee", RAMA)
    os.chdir("/content/ia-Kaggle-Coffee")
    print("Rama:", usada)

# Subir hasta la carpeta del proyecto (con src/ y scripts/), desde donde se abra el notebook (PyCharm/Jupyter).
raiz = Path.cwd()
while not ((raiz / "src").is_dir() and (raiz / "scripts" / "entrenar_v5.py").is_file()) and raiz != raiz.parent:
    raiz = raiz.parent
if not (raiz / "scripts" / "entrenar_v5.py").is_file():
    raise SystemExit("No encuentro scripts/entrenar_v5.py. Abra el notebook dentro del repositorio (rama claude/modelo-v5).")
os.chdir(raiz)
sys.path.insert(0, str(raiz))
print("Proyecto:", raiz)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

## Dependencias y GPU
En Colab **no** se reinstala TensorFlow (se usa el que trae, con Keras 3 y CUDA ya configurados); solo se instala lo que falte para exportar a ONNX.

In [ ]:
import importlib

PINES = {"onnx": "onnx==1.22.0", "tf2onnx": "tf2onnx==1.17.0", "onnxruntime": "onnxruntime==1.32.0",
         "sklearn": "scikit-learn", "pandas": "pandas", "h5py": "h5py"}
faltan = []
for modulo, paquete in PINES.items():
    try:
        importlib.import_module(modulo)
    except ImportError:
        faltan.append(paquete)
try:
    importlib.import_module("tensorflow")
except ImportError:
    faltan.append("tensorflow==2.19.1")           # solo fuera de Colab
if faltan:
    print("Instalando:", faltan)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)

versiones = subprocess.run([sys.executable, "-c",
    "import sys, tensorflow as tf, keras, onnx, tf2onnx, onnxruntime, sklearn; "
    "print('Python', sys.version.split()[0], '| TF', tf.__version__, '| Keras', keras.__version__, '| onnx', onnx.__version__, "
    "'| tf2onnx', tf2onnx.__version__, '| onnxruntime', onnxruntime.__version__, '| sklearn', sklearn.__version__); "
    "print('GPU:', tf.config.list_physical_devices('GPU'))"], capture_output=True, text=True)
print(versiones.stdout.strip() or versiones.stderr[-2000:])
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True) \
    if subprocess.run(["which", "nvidia-smi"], capture_output=True).returncode == 0 else None
if smi and smi.returncode == 0:
    print("GPU:", smi.stdout.strip())
else:
    print("ATENCIÓN: no hay GPU. Entorno de ejecución > Cambiar tipo de entorno > A100 o L4. "
          "Sin GPU el protocolo completo tarda muchas horas." + ("" if FAST else " (con FAST = True sí se puede probar en CPU)"))

## Google Drive (checkpoints reanudables)

In [ ]:
DIR_DRIVE = None
if EN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DIR_DRIVE = Path("/content/drive/MyDrive/cafe_v5")
    except Exception as e:
        print("No se pudo montar Drive; se usará el disco temporal de Colab (se pierde al desconectar):", e)
BASE = DIR_DRIVE or (Path("/content/cafe_v5") if EN_COLAB else raiz / "checkpoints" / "cafe_v5")
BASE.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = BASE / "checkpoints"
sufijo = "_rapido" if FAST else ""
SALIDA = (BASE / "web_model_rapido") if FAST else (raiz / "web" / "model")
RESULTADOS = (BASE / "results_rapido") if FAST else (raiz / "results" / "v5")
ZIP = BASE / f"modelo_cafe_v5{sufijo}.zip"
print("Checkpoints:", CHECKPOINT_DIR, "\nExportación:", SALIDA, "\nResultados:", RESULTADOS, "\nPaquete:", ZIP)

## Protocolo completo
Cada semilla terminada queda guardada en Drive. Si se corta, vuelva a ejecutar: las semillas hechas se reutilizan, la decisión y el test (una sola vez) también.

In [ ]:
cmd = [sys.executable, "-u", "scripts/entrenar_v5.py", "--backbone", BACKBONE, "--checkpoints", str(CHECKPOINT_DIR),
       "--salida", str(SALIDA), "--resultados", str(RESULTADOS), "--zip", str(ZIP)]
if FAST:
    cmd.append("--rapido")
print(" ".join(cmd))
entorno = {**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"}
inicio = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=entorno)
for linea in proc.stdout:
    print(linea, end="")
proc.wait()
print(f"\nDuración: {(time.time() - inicio) / 60:.1f} min")
if proc.returncode != 0:
    raise SystemExit(f"entrenar_v5.py terminó con código {proc.returncode}. Revise el mensaje de arriba; "
                     "lo ya entrenado quedó en Drive: al volver a ejecutar continúa.")

## Resumen

In [ ]:
import json
import pandas as pd

corridas = pd.read_csv(RESULTADOS / "corridas_v5.csv", index_col="semilla")
resumen = json.loads((RESULTADOS / "resumen_v5.json").read_text())
test = json.loads((RESULTADOS / "test_v5.json").read_text())
card = json.loads((SALIDA / "model_card.json").read_text())
onnx_v = json.loads((RESULTADOS / "onnx_verificacion.json").read_text())

print("VALIDACIÓN por semilla")
print(corridas[["f1_macro", "accuracy", "f1_healthy", "f1_red_spider_mite", "f1_rust", "f1_macro_tta_flip", "segundos"]].round(4).to_string())
tabla = pd.DataFrame({
    "val F1 macro": [f"{resumen['val_f1_macro_media']:.3f} ± {resumen['val_f1_macro_desv']:.3f}", f"{resumen['val_f1_macro_mejor']:.3f}",
                     f"{resumen['val_f1_macro_conjunto']:.3f}"],
    "val F1 macro TTA (volteo)": ["", f"{resumen['val_f1_macro_mejor_tta_flip']:.3f}", f"{resumen['val_f1_macro_conjunto_tta_flip']:.3f}"],
    "test F1 macro": ["", f"{test['individual']['f1_macro']:.3f}", f"{test['conjunto']['f1_macro']:.3f}"],
    "test exactitud": ["", f"{test['individual']['accuracy']:.3f}", f"{test['conjunto']['accuracy']:.3f}"],
}, index=["media de semillas", f"mejor individual (semilla {resumen['mejor_semilla']})", "conjunto de semillas"])
print("\n" + tabla.to_string())
print(f"\nSe exportó: {resumen['exportar']}  |  versión {card['version']}  |  Keras vs ONNX (fotos reales): {onnx_v['max_dif_fotos_reales']:.1e}")
print("Modelo anterior (4b-2026-11-04, otros splits): test exactitud 0,60, F1 macro 0,53.")

## Paquete para la web
`modelo_cafe_v5.zip` contiene `web/model/` (cafe_robusta.onnx, labels.json, model_card.json, paridad.json, metricas_web.json) y los resultados (`results/v5/`). Queda en `MyDrive/cafe_v5/` y se descarga.

In [ ]:
print("Paquete:", ZIP, f"({ZIP.stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(str(ZIP))
except ImportError:
    pass

## Qué hacer con el resultado
- Descomprima el zip en la raíz del repositorio (reemplaza `web/model/*` y crea `results/v5/`), ejecute `cd web && npx vitest run` y súbalo.
- Si el F1 macro de **test** no mejora claramente al modelo actual, compárelo con cuidado: los splits v5 son distintos (más limpios), así que las cifras no son directamente comparables.
- Pendiente: validar con café arábica y con fotos de cultivos colombianos.